In [1]:
import pandas as pd

In [37]:
df = pd.read_csv("glamandglow.csv")

In [25]:
df.shape

(1001, 31)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 31 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   Product ID                                                      1001 non-null   object 
 1   Product Name                                                    1001 non-null   object 
 2   Product Type                                                    1001 non-null   object 
 3   Category                                                        1001 non-null   object 
 4   Brand Name                                                      1001 non-null   object 
 5   Product Line Name                                               0 non-null      float64
 6   Ingredients                                                     921 non-null    object 
 7   Use Instructions                                   

### Filter out Product Names

In [5]:
df = df.dropna(subset=["Product Name"])

In [53]:
print(df["Source"][10])

https://glamandglowbeautyhub.shop/collections/all/products/tree-hut-vitamin-c-shea-sugar-scrub


In [6]:
df = df.dropna(subset=["Product Name"])

### Category 

In [42]:
### product category


# --- Define product type groups ---
product_type_groups = {
    "Skin Care": [
        "Ointments", "Creams", "Lotions", "Gels", "Pastes", "Balms", "Solutions",
        "Serums", "Mists", "Foams", "Powders", "Sticks", "Transdermal Patches",
        "Masks", "Emulgels", "Nanoemulsions", "Microsponges", "Liposomes",
        "Niosomes", "Hydrogels", "Sunscreen", "Cleansing Balms", "Cleansing Oils",
        "Body Butters", "Paints", "Suspensions", "Cleansers", "Toners",
        "Exfoliators", "Chemical Peels", "Eye Creams", "Moisturizers", 
        "Spot Treatments", "Face Oils","Scrubs","Body Treatments","Oils"
    ],

    "Baby Care": [
        "Powders", "Shampoos", "Washes", "Gels", "Wipes", "Diaper Rash Creams",
        "Bubble Bath", "Baby Balm", "Mineral Sunscreen"
    ],

    "Men's Care": [
        "Shaving Creams", "Shaving Gels", "Shaving Foams", "Aftershave Balms",
        "Aftershave Lotions", "Aftershave Gels", "Beard Oils", "Beard Waxes",
        "Face Washes", "Face Scrubs", "Toners", "Serums", "Moisturizers",
        "Eye Creams", "Lip Balms", "Body Washes", "Deodorants",
        "Hair Gels", "Hair Waxes", "Hair Sprays", "Face Oils", "Pomades",
        "Hair Conditioners", "Hair Treatments"
    ],

    "Supplements": [
        "Collagen", "Vitamin C", "Vitamin A", "Vitamin E", "Vitamin D", "Biotin", 
        "Zinc", "Selenium", "Copper", "Omega-3 and Omega-6 Fatty Acids", 
        "Hyaluronic Acid", "Coenzyme Q10", "Polypodium Leucotomos Extract", 
        "Resveratrol", "Probiotics"
    ],

    "Injectables": [
        "Botox", "Dysport", "Xeomin", "Jeuveau", "Daxxify", "Juvéderm", 
        "Restylane", "Belotero", "Radiesse", "Sculptra", "Bellafill",
        "Autologous Fat Injections", "Kybella", "Platelet-Rich Plasma (PRP) Therapy",
        "Injectable Skin Boosters", "Profhilo", "Jalupro"
    ]
}


import pandas as pd
import inflect

p = inflect.engine()

# Normalize singular/plural terms
def normalize_terms(terms):
    normalized = set()
    for term in terms:
        t = term.lower()
        normalized.add(t)
        singular = p.singular_noun(t)
        if singular:
            normalized.add(singular.lower())
        else:
            plural = p.plural(t)
            if plural:
                normalized.add(plural.lower())
    return normalized

# Build normalized dictionary
normalized_groups = {
    category: normalize_terms(terms)
    for category, terms in product_type_groups.items()
}

import re

def clean_text(text):
    """Remove leading/trailing whitespace and normalize inner spaces."""
    if not text or pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", text.strip())

import re

# Category priority (highest → lowest)
category_priority = ["Skin Care", "Men's Care", "Baby Care", "Supplements", "Injectables"]

def get_category_and_type(row):
    product_name = clean_text(row.get("Product Name", ""))
    description = row.get("Description", "")

    if pd.notna(description):
        desc_lines = str(description).splitlines()[:2]
        description = clean_text(" ".join(desc_lines))
    else:
        description = ""

    text_to_search = (product_name + " " + description).strip().lower()
    if not text_to_search:
        return pd.Series(["Other", "Other"], index=["Category", "Product Type"])

    for cat in category_priority:
        for keyword in normalized_groups[cat]:
            pattern = r"\b" + re.escape(keyword) + r"\b"
            if re.search(pattern, text_to_search):
                return pd.Series([keyword, cat], index=["Category", "Product Type"])

    return pd.Series(["Other", "Other"], index=["Category", "Product Type"])


# Apply to DataFrame
# Apply function
df[["Category", "Product Type"]] = df.apply(get_category_and_type, axis=1)

In [44]:
print(df["Product Name"][:12])

0                          SALTAIR EXOTIC PULP BODY OIL
1     GOOD MOLECULES DISCOLORATION CORRECTING BODY T...
2         FEMFRESH INTIMATE HYGIENE - 0% SENSITIVE WASH
3     OLAY FRESH OUTLAST BODY WASH - BIRCH WATER & L...
4       VICTORIA'S SECRET - BARE VANILLA FRAGRANCE MIST
5                  B_LAB MATCHA HYDRATING FOAM CLEANSER
6          ACWELL LICORICE PH BALANCING CLEANSING TONER
7     BLISS BRIGHT IDEA VITAMIN C + TRI-PEPTIDE BRIG...
8                   TREE HUT VITAMIN C SHEA SUGAR SCRUB
9     ROC SKINCARE RETINOL CORREXION LINE SMOOTHING ...
10     NIVEA CRÈME BODY, FACE & HAND MOISTURIZING CREAM
11    YOUTH TO THE PEOPLE RETINAL + NIACINAMIDE YOUT...
Name: Product Name, dtype: object


### Get Product Type by category

In [46]:
print(df["Category"][:12])

0                oil
1     body treatment
2               wash
3          body wash
4               mist
5               foam
6              toner
7              serum
8              scrub
9              serum
10             cream
11             serum
Name: Category, dtype: object


### Filter out non-skin care form

In [29]:
# Filter out rows where Category is "Other"
df = df[df["Category"] != "Other"].reset_index(drop=True)

### Ingredients

In [9]:
import pandas as pd
import logging

# --- Setup logging ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# --- Load ingredient dictionary ---
dictiona = pd.read_csv("ingred_dict.csv")

name_cols = ["INCI name", "Chem/IUPAC Name / Description"]

# Flatten into one big Series
all_names = pd.Series(dtype=str)
for col in name_cols:
    if col in dictiona.columns:
        all_names = pd.concat([all_names, dictiona[col].dropna()])

# Standardize for matching
known_ingredients = set(all_names.str.strip().str.upper().unique())
logging.info(f"✅ Loaded {len(known_ingredients)} unique ingredient names")

# --- Helper function ---
def extract_ingredients(text, row_id, source_col):
    """Extract ingredients from text using dictionary lookup."""
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        logging.info(f"[Row {row_id}] {source_col}: empty, 0 matches")
        return ""
    
    text_upper = text.upper()
    found = [ing for ing in known_ingredients if ing in text_upper]

    logging.info(f"[Row {row_id}] {source_col}: found {len(found)} match(es)")
    return ", ".join(sorted(set(found)))


# --- Extract from description ---
df["Ingredients_From_Description"] = df.apply(
    lambda row: extract_ingredients(row.get("Product Description", ""), row.name, "Description"),
    axis=1
)

# --- Combine all ingredient sources (only one for now) ---
def combine_ingredients(row):
    """Merge ingredient matches from multiple sources into one field."""
    sources = [row.get("Ingredients_From_Description", "")]
    ingredients = set()

    for source in sources:
        if source:
            ingredients.update([s.strip() for s in source.split(",") if s.strip()])

    return ", ".join(sorted(ingredients)) if ingredients else ""

df["Ingredients"] = df.apply(combine_ingredients, axis=1)


2025-09-12 01:45:56,822 | INFO | ✅ Loaded 55674 unique ingredient names
2025-09-12 01:45:56,866 | INFO | [Row 0] Description: found 5 match(es)
2025-09-12 01:45:56,908 | INFO | [Row 1] Description: found 26 match(es)
2025-09-12 01:45:56,957 | INFO | [Row 2] Description: found 1 match(es)
2025-09-12 01:45:57,016 | INFO | [Row 3] Description: found 26 match(es)
2025-09-12 01:45:57,062 | INFO | [Row 4] Description: found 3 match(es)
2025-09-12 01:45:57,087 | INFO | [Row 5] Description: found 2 match(es)
2025-09-12 01:45:57,191 | INFO | [Row 6] Description: found 44 match(es)
2025-09-12 01:45:57,241 | INFO | [Row 7] Description: found 3 match(es)
2025-09-12 01:45:57,301 | INFO | [Row 8] Description: found 7 match(es)
2025-09-12 01:45:57,406 | INFO | [Row 9] Description: found 4 match(es)
2025-09-12 01:45:57,454 | INFO | [Row 10] Description: found 2 match(es)
2025-09-12 01:45:57,520 | INFO | [Row 11] Description: found 10 match(es)
2025-09-12 01:45:57,557 | INFO | [Row 12] Description: fou

### Use Instructions and Benefits

In [10]:
import re
import pandas as pd

# --- Keywords ---
use_keywords = [
    "Directions for use", "Usage instructions", "Application guidelines",
    "Instructions for application", "Method of use", "Directions for application",
    "How to apply", "Application steps", "Application method", "Usage directions",
    "Guidelines for use", "Instructions", "Directions", "Method", "Procedure",
    "Guide", "Manual", "Handbook", "Tutorial", "Operating instructions",
    "User guide"
]

benefit_keywords = [
    "key features", "benefits", "attributes", "characteristics", "properties",
    "specifications", "functions", "components", "elements", "qualities",
    "hallmarks", "highlights", "specifics", "capabilities", "traits",
    "aspects", "distinctions", "advantages", "gains", "rewards", "profits",
    "returns", "upsides", "perks", "privileges", "merits", "yields",
    "payoffs", "value", "plus points", "favorable outcomes", "contribut",
    "features & details"   # ✅ included
]

benefit_stop_keywords = [
    "about this item", "product description", "ingredients"
]

# lowercase for matching
use_keywords = [kw.lower() for kw in use_keywords]
benefit_keywords = [kw.lower() for kw in benefit_keywords]
benefit_stop_keywords = [kw.lower() for kw in benefit_stop_keywords]


def strip_keyword(block: str, keywords: list):
    """Remove only the keyword itself, preserve following text."""
    block_lower = block.lower()
    for kw in keywords:
        if block_lower.startswith(kw):
            # length of keyword in original block
            kw_len = len(block[:len(kw)])
            # strip just the keyword and optional separators
            cleaned = block[kw_len:].lstrip(" :-–\n\t")
            return cleaned.strip()
    return block.strip()



def split_sections(text):
    if not isinstance(text, str) or not text.strip():
        return "", "", ""

    desc = text.strip()
    desc_lower = desc.lower()

    use_block, benefit_blocks = "", []

    # --- Step 1: Extract Use Instructions ---
    use_pos = [(desc_lower.find(kw), kw) for kw in use_keywords if kw in desc_lower]
    if use_pos:
        start_pos, kw = min(use_pos)
        after_text = desc_lower[start_pos:]
        stop_pos = None
        for stop_kw in benefit_keywords:
            idx = after_text.find(stop_kw)
            if idx > 0:
                stop_pos = start_pos + idx
                break
        if stop_pos:
            raw_block = desc[start_pos:stop_pos].strip()
            use_block = strip_keyword(raw_block, use_keywords)
            desc = (desc[:start_pos] + desc[stop_pos:]).strip()
            desc_lower = desc.lower()
        else:
            raw_block = desc[start_pos:].strip()
            use_block = strip_keyword(raw_block, use_keywords)
            desc = desc[:start_pos].strip()
            desc_lower = desc.lower()

    # --- Step 2: Extract ALL Benefit Sections ---
    while True:
        ben_pos = [(desc_lower.find(kw), kw) for kw in benefit_keywords if kw in desc_lower]
        ben_pos = [(pos, kw) for pos, kw in ben_pos if pos >= 0]

        if not ben_pos:
            break

        start_pos, kw = min(ben_pos)
        after_text = desc_lower[start_pos:]

        stop_pos = None
        for stop_kw in benefit_stop_keywords:
            idx = after_text.find(stop_kw)
            if idx > 0:
                stop_pos = start_pos + idx
                break

        if stop_pos:
            raw_block = desc[start_pos:stop_pos].strip()
            if "ingredients" not in raw_block.lower():
                benefit_blocks.append(strip_keyword(raw_block, benefit_keywords))
            desc = (desc[:start_pos] + desc[stop_pos:]).strip()
            desc_lower = desc.lower()
        else:
            raw_block = desc[start_pos:].strip()
            if "ingredients" not in raw_block.lower():
                benefit_blocks.append(strip_keyword(raw_block, benefit_keywords))
            desc = desc[:start_pos].strip()
            desc_lower = desc.lower()

    return desc.strip(), use_block, " ".join(benefit_blocks)


# --- Apply to DataFrame ---
df[["Product Description", "Use Instructions", "Benefits"]] = df["Product Description"].apply(
    lambda x: pd.Series(split_sections(x))
)

In [18]:
df.to_csv("try.csv",index=False)

### Clean up Product Description

In [19]:
import re

def clean_product_description(text: str) -> str:
    """
    Remove unwanted prefixes like 'Item Description', 'About this item',
    'Product Description', 'Product Details' from the beginning of description.
    """
    if not isinstance(text, str) or not text.strip():
        return text
    
    # Define unwanted headers (case-insensitive)
    unwanted_prefixes = [
        r"item description",
        r"about this item",
        r"product description",
        r"product details"
    ]
    
    # Build regex to match any of them at the beginning, allowing punctuation/colon after
    pattern = r"^\s*(?:" + "|".join(unwanted_prefixes) + r")\s*[:\-–]*\s*"
    
    cleaned = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return cleaned.strip()

In [16]:
df["Product Description"] = df["Product Description"].apply(clean_product_description)

### Package size clean up

In [12]:
import re

def clean_package_size(text: str) -> str:
    """
    Validate and clean package size values.
    Keeps single or comma-separated valid sizes (weight, volume, counts).
    Invalid entries are removed.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # Regex for valid package sizes
    pattern = re.compile(
        r"^\s*(\d+(\.\d+)?\s*(mg|g|kg|ml|l|oz|fl oz|mcg|iu|units?|capsules?|tablet[s]?|tabs?|pcs?|pieces?|pack[s]?|bottle[s]?|vial[s]?))\s*$",
        flags=re.IGNORECASE
    )
    
    # Split on commas, clean each token
    parts = [p.strip() for p in text.split(",")]
    valid_parts = [p for p in parts if pattern.match(p)]
    
    return ", ".join(valid_parts) if valid_parts else ""

In [13]:
df["Package Size"] = df["Package Size"].apply(clean_package_size)

### Brand

In [15]:

# Load your cosmetics dataset
df1 = pd.read_csv("cosmetics.csv")

# Build dictionary of known brands
brand_dict = set(df1["Brand"].dropna().str.upper().unique())


In [16]:
import re

def extract_brand(product_name, brand_dict):
    name_upper = str(product_name).upper()

    # Try dictionary match
    for brand in brand_dict:
        if brand in name_upper:
            return brand.title()

    # --- Fallback heuristics ---
    # Rule 1: First 2 words are often the brand
    candidate = " ".join(name_upper.split()[:2])

    # Rule 2: If candidate has + or -, strip after
    candidate = re.split(r"[-+]", candidate)[0].strip()

    return candidate.title()


In [17]:
df["Brand Name"] = df["Product Name"].apply(lambda x: extract_brand(x, brand_dict))

### Product Id

In [20]:
df['Product ID'] = [f"Vef_GLAMANDGLOW_{i+1}" for i in range(len(df))]

In [47]:
df.to_csv("glamandglow.csv",index=False)

### Download Images

In [23]:
import os
import re
import requests
import pandas as pd
from time import sleep
from random import uniform
from urllib.parse import urlparse
from PIL import Image
from io import BytesIO

# --- File paths ---
INPUT_FILE = "processed.csv"                  # Your CSV input file
CHECKPOINT_FILE = "products_with_images.csv"  # Backup file for resuming
IMAGES_FOLDER = "images"

# --- Setup ---
os.makedirs(IMAGES_FOLDER, exist_ok=True)

# ---------------- Helpers ---------------- #
def get_headers(url):
    parsed = urlparse(url)
    domain = f"{parsed.scheme}://{parsed.hostname}" if parsed.scheme else ""
    return {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0.0.0 Safari/537.36"
        ),
        "Referer": domain,
    }

def clean_url(url: str):
    """Normalize image URLs (remove junk, force https)."""
    if not url or not isinstance(url, str):
        return None
    url = url.strip()
    if not url.startswith("http"):
        url = "https://" + url.lstrip("/")
    url = url.split("?")[0]
    url = re.sub(r"(_(small|compact|medium|large|grande|x\d+))", "_master", url)
    return url

def download_image(url, filename):
    """Download a single image and save as JPEG."""
    try:
        url = clean_url(url)
        if not url:
            return None

        filepath = os.path.join(IMAGES_FOLDER, f"{filename}.jpg")
        if os.path.exists(filepath):
            return os.path.basename(filepath)  # Already downloaded

        response = requests.get(url, headers=get_headers(url), timeout=30)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))

        # Handle transparency (convert to white background)
        if image.mode in ("RGBA", "LA", "P"):
            bg = Image.new("RGB", image.size, (255, 255, 255))
            image = image.convert("RGBA")
            bg.paste(image, mask=image.split()[-1])
            image = bg
        else:
            image = image.convert("RGB")

        image.save(filepath, "JPEG", quality=95)
        return os.path.basename(filepath)

    except Exception as e:
        print(f"❌ Failed to download {url}: {e}")
        return None

def process_row_images(url_list, barcode, existing_files=None):
    """
    Download up to 2 images for each product, saved under its Barcode (EAN/UPC).
    """
    if pd.isna(url_list):
        return None

    # Split multiple URLs into list
    cleaned = str(url_list).strip().strip("[]").replace("'", "").replace('"', "")
    urls = [u.strip() for u in re.split(r",\s*", cleaned) if u.strip()]

    # Filter unwanted images
    urls = [u for u in urls if not any(bad in u.lower() for bad in ["logo", "icon", "placeholder"])]

    # Keep only first 2 images
    urls = urls[:2]

    if not urls:
        return None

    saved_files = []

    # Always clean barcode (though you said they're normalized)
    base_name = str(barcode).strip() if barcode and str(barcode).strip() else "NO_BARCODE"

    for i, url in enumerate(urls, start=1):
        filename = f"{base_name}_{i}" if len(urls) > 1 else base_name

        # Skip if file already exists
        if existing_files:
            expected_file = f"{filename}.jpg"
            if expected_file in existing_files and os.path.exists(os.path.join(IMAGES_FOLDER, expected_file)):
                saved_files.append(expected_file)
                continue

        saved = download_image(url, filename)
        if saved:
            saved_files.append(saved)

        # Polite random delay
        sleep(uniform(0.2, 0.6))

    return ", ".join(saved_files) if saved_files else None

# ---------------- Main ---------------- #
if os.path.exists(CHECKPOINT_FILE):
    print(f"📂 Resuming from {CHECKPOINT_FILE}")
    df = pd.read_csv(CHECKPOINT_FILE, dtype=str)
else:
    print(f"📥 Loading fresh data from {INPUT_FILE}")
    df = pd.read_csv(INPUT_FILE, dtype=str)

# Reset Product Images column to empty
df["Product Images"] = ""


# Ensure column exists
if "Product Images" not in df.columns:
    df["Product Images"] = None
df["Product Images"] = df["Product Images"].astype("object")

SAVE_EVERY = 20

for idx, row in df.iterrows():
    barcode = row.get("Product ID")
    existing_files = None

    # Re-use existing images if still on disk
    if pd.notna(row["Product Images"]) and row["Product Images"]:
        existing_files = [f.strip() for f in str(row["Product Images"]).split(",") if f.strip()]
        missing = [f for f in existing_files if not os.path.exists(os.path.join(IMAGES_FOLDER, f))]
        if not missing:
            continue
        else:
            print(f"⚠️ Missing files for Barcode {barcode}, re-downloading...")

    df.at[idx, "Product Images"] = process_row_images(
        row.get("Product Image URL"), barcode, existing_files
    )

    if idx % SAVE_EVERY == 0:
        df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"💾 Checkpoint saved at row {idx}")

# Final save
df.to_csv(CHECKPOINT_FILE, index=False)
print(f"🎉 Finished! All images processed. Backup saved to {CHECKPOINT_FILE}")

📥 Loading fresh data from processed.csv
💾 Checkpoint saved at row 0
💾 Checkpoint saved at row 20
💾 Checkpoint saved at row 40
💾 Checkpoint saved at row 60
❌ Failed to download https://glamandglowbeautyhub.shop/cdn/shop/files/SLEEVE_NEOPROSONEVITCFOAMING_FRONT_master_2x_jpg_2048x.webp: 404 Client Error: Not Found for url: https://glamandglowbeautyhub.shop/cdn/shop/files/SLEEVE_NEOPROSONEVITCFOAMING_FRONT_master_2x_jpg_2048x.webp
💾 Checkpoint saved at row 80
💾 Checkpoint saved at row 100
💾 Checkpoint saved at row 120
💾 Checkpoint saved at row 140
💾 Checkpoint saved at row 160
💾 Checkpoint saved at row 180
💾 Checkpoint saved at row 200
💾 Checkpoint saved at row 220
💾 Checkpoint saved at row 240
💾 Checkpoint saved at row 260
💾 Checkpoint saved at row 280
💾 Checkpoint saved at row 300
❌ Failed to download https://glamandglowbeautyhub.shop/cdn/shop/files/TamarindLotioncopy_master_2x_17815c2f-1a5d-42b5-9b9d-f20e0c15e647_2048x.webp: 404 Client Error: Not Found for url: https://glamandglowbeaut